In [ ]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
load_dotenv()
llmModel = ChatGroq(model='openai/gpt-oss-120b')
openrouter_api_key = os.environ["OPENROUTER_API_KEY"]

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
loader = PyPDFLoader("Res.pdf")
docs = loader.load()

In [ ]:
# Bydefault Langchain loads your files in Document format
docs

In [ ]:
len(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [ ]:
len(chunks)

When a user asks a question in text, the retriever performs a three-step matching process behind the scenes:

1.Translation: It automatically runs the user's text question through the exact same embedding model used to build the database. 

2.Comparison: FAISS takes the question's numbers and compares them against all the numbers stored in the database using math.

3.Retrieval: Once FAISS identifies the stored numbers that are the closest mathematical match to the question's numbers, it pulls out the text chunks attached to them and returns them to the AI.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=openrouter_api_key,
    model="nvidia/nemotron-3-embed-1b:free",
    check_embedding_ctx_length=False # skips that check and sends your plain text straight to OpenRouter
)

In [ ]:
embeddings = embeddings_model

vector_db = FAISS.from_documents(chunks, embeddings)
vector_db.save_local("faiss_DB")

In [ ]:
vector_db

In [ ]:
retriever = vector_db.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [ ]:
response = retriever.invoke("What is Word lookup Dictionary?")
response

""" : A docstring (a block of text inside triple quotes that describes what a function does). The AI reads this description to decide when it should use this tool.

In [ ]:
@tool
def rag_tool(query: str) -> str:
    """
    Retrieve relevant information from the PDF document.

    Use this tool when the user asks factual or conceptual questions
    that may be answered using the stored PDF documents.

    Args:
        query: The question or search query used to retrieve PDF content.
    """

# every retriever (a tool that searches a database for matching text) is built to always return a list of Document
    documents = retriever.invoke(query)

    if not documents:
        return "No relevant information was found in the PDF."

    formatted_documents = []

# A loop that goes through each matching text piece one by one [document] and numbers them [index] starting from 1.
    for index, document in enumerate(documents, start=1):
        source = document.metadata.get("source", "Unknown source") # Extracts metadata
        page = document.metadata.get("page", "Unknown page")       # Extracts metadata

# Combines the FINAL result number, file name, page number, and page_content
        formatted_documents.append(
            f"Document {index}\n"
            f"Source: {source}\n"
            f"Page: {page}\n"
            f"Content: {document.page_content}"
        )

    return "\n\n".join(formatted_documents)

In [ ]:
tools = [rag_tool]
llm_with_tools = llmModel.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
# graph nodes
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)  # Executes tool calls

In [ ]:
# graph structure
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "chat_node")

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")  
chatbot = graph.compile()

chatbot

In [ ]:
# Regular chat
result = chatbot.invoke({"messages": [HumanMessage(content="Hello")]})

print(result["messages"][-1].content)

In [ ]:
# Regular chat
result = chatbot.invoke({"messages": [HumanMessage(content="Using the pdf notes,What is Word lookup Dictionary?")]})

print(result["messages"][-1].content)